# 8.2 Learning Rate

本 notebook 使用合成二元分類資料，比較固定 learning rate、learning rate schedule 與 `ReduceLROnPlateau`。重點是觀察 learning curve，而不是只看最後一個 accuracy。


## 1. 載入套件


In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)


## 2. 建立資料

`make_moons` 是非線性二元分類資料，很適合觀察不同 learning rate 對 DNN 訓練穩定度的影響。


In [ ]:
X, y = make_moons(n_samples=1800, noise=0.25, random_state=SEED)
X = X.astype('float32')
y = y.astype('float32')

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print(x_train.shape, x_valid.shape, x_test.shape)
print(pd.Series(y_train).value_counts().sort_index())


## 3. 建立共用模型函式

後續所有實驗都使用同一個模型架構，只改 optimizer 的 learning rate 設定。


In [ ]:
def build_model():
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    return model


def train_with_optimizer(name, optimizer, callbacks=None, epochs=35):
    model = build_model()
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_valid, y_valid),
        epochs=epochs,
        batch_size=32,
        callbacks=callbacks,
        verbose=0,
    )
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    return {
        'name': name,
        'model': model,
        'history': history,
        'test_loss': test_loss,
        'test_accuracy': test_acc,
    }


## 4. 比較固定 Learning Rate


In [ ]:
fixed_results = []
for lr in [1e-4, 1e-3, 1e-2]:
    result = train_with_optimizer(
        name=f'fixed_lr={lr:g}',
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        epochs=35,
    )
    fixed_results.append(result)

summary = pd.DataFrame([
    {
        'setting': r['name'],
        'best_val_accuracy': max(r['history'].history['val_accuracy']),
        'final_val_loss': r['history'].history['val_loss'][-1],
        'test_accuracy': r['test_accuracy'],
    }
    for r in fixed_results
]).sort_values('best_val_accuracy', ascending=False)
summary


## 5. 視覺化 Learning Curve


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for r in fixed_results:
    plt.plot(r['history'].history['val_loss'], label=r['name'])
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
for r in fixed_results:
    plt.plot(r['history'].history['val_accuracy'], label=r['name'])
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


## 6. 使用 ExponentialDecay Schedule

`ExponentialDecay` 讓 learning rate 隨訓練步數逐漸下降，常用在一開始希望快速學習、後期希望細緻收斂的情境。


In [ ]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-2,
    decay_steps=40,
    decay_rate=0.85,
    staircase=True,
)

schedule_result = train_with_optimizer(
    name='exponential_decay',
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    epochs=35,
)

print('Best val accuracy:', max(schedule_result['history'].history['val_accuracy']))
print('Test accuracy:', schedule_result['test_accuracy'])


## 7. 使用 ReduceLROnPlateau

`ReduceLROnPlateau` 會在 validation loss 停滯時降低 learning rate，適合不確定 schedule 節奏時使用。


In [ ]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=0,
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
)

plateau_result = train_with_optimizer(
    name='reduce_on_plateau',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
    callbacks=[reduce_lr, early_stop],
    epochs=45,
)

print('Epochs:', len(plateau_result['history'].history['loss']))
print('Best val accuracy:', max(plateau_result['history'].history['val_accuracy']))
print('Test accuracy:', plateau_result['test_accuracy'])


## 8. 彙整比較


In [ ]:
all_results = fixed_results + [schedule_result, plateau_result]
comparison = pd.DataFrame([
    {
        'setting': r['name'],
        'epochs': len(r['history'].history['loss']),
        'best_val_accuracy': max(r['history'].history['val_accuracy']),
        'best_val_loss': min(r['history'].history['val_loss']),
        'test_accuracy': r['test_accuracy'],
    }
    for r in all_results
]).sort_values('best_val_accuracy', ascending=False)
comparison


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for r in all_results:
    plt.plot(r['history'].history['val_loss'], label=r['name'])
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
for r in all_results:
    plt.plot(r['history'].history['val_accuracy'], label=r['name'])
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


## 9. 套用自己的資料

替換成自己的資料時，建議先固定模型架構、batch size 與資料切分，只比較 learning rate。找到合理範圍後，再加入 schedule 或 `ReduceLROnPlateau`。如果數值特徵尺度差異很大，請先做標準化，否則 learning rate 實驗會很不穩定。
